# EDA — Zepto Inventory Dataset
This notebook walks through a concise exploratory data analysis: loading data, quick cleaning, key aggregates, and visualizations. It is interview-friendly: each section states the business question, the SQL/pandas approach, and the takeaway.

Files used: `zepto_v2.csv`, `sql/queries.sql` (examples). Generated plots are saved to `../plots/`.

In [1]:
# 1) Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
sns.set(style='whitegrid')
DATA_PATH = os.path.join('..','zepto_v2.csv')
PLOTS_DIR = os.path.join('..','plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

In [2]:
# 2) Load data
df = pd.read_csv(DATA_PATH, encoding='utf-8')
# normalize column names to snake_case lower for stable access
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()
df.shape

(3732, 9)

In [3]:
# Quick overview: head, dtypes, nulls
display(df.head(6))
print(df.dtypes)
print('Nulls:')
print(df.isnull().sum())

,category,name,mrp,discountpercent,availablequantity,discountedsellingprice,weightingms,outofstock,quantity
0,Fruits & Vegetables,Onion,2500,16,3,2100,1000,False,1
1,Fruits & Vegetables,Tomato Hybrid,4200,16,3,3500,1000,False,1
2,Fruits & Vegetables,Tender Coconut,5100,15,3,4300,58,False,1
3,Fruits & Vegetables,Coriander Leaves,2000,15,3,1700,100,False,100
4,Fruits & Vegetables,Ladies Finger,1400,14,3,1200,250,False,250
5,Fruits & Vegetables,Potato,3500,17,3,2900,1000,False,1


category                    str
name                        str
mrp                       int64
discountpercent           int64
availablequantity         int64
discountedsellingprice    int64
weightingms               int64
outofstock                 bool
quantity                  int64
dtype: object
Nulls:
category                  0
name                      0
mrp                       0
discountpercent           0
availablequantity         0
discountedsellingprice    0
weightingms               0
outofstock                0
quantity                  0
dtype: int64


## Cleaning
Business question: ensure prices are sensible and comparable.
Steps: coerce numerics, drop zero-priced rows, convert paise→rupees if detected, compute price-per-gram.

In [4]:
# Cleaning steps
numeric_cols = ['mrp','discountpercent','discountedsellingprice','availablequantity','weightingms','quantity']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
# drop rows missing both prices
df = df[~((df['mrp'].isna()) & (df['discountedsellingprice'].isna()))]
# drop zero priced rows
df = df[~(df['mrp'] == 0)]
df = df[~(df['discountedsellingprice'] == 0)]
# paise->rupees conversion if needed
if df['mrp'].max() > 10000:
    df['mrp'] = df['mrp'] / 100.0
    df['discountedsellingprice'] = df['discountedsellingprice'] / 100.0
# price per gram
if 'weightingms' in df.columns:
    df['price_per_gram'] = df['discountedsellingprice'] / df['weightingms'].replace({0: pd.NA})
df.shape

(3731, 10)

In [5]:
# 4) Key aggregates
print('Total records:', len(df))
if 'category' in df.columns:
    print('Distinct categories:', df['category'].nunique())
if 'outofstock' in df.columns:
    print('In-stock vs out-of-stock:\n', df['outofstock'].value_counts(dropna=False))
# Top discounts
display(df.sort_values('discountpercent', ascending=False)[['name','discountpercent','discountedsellingprice']].head(10))

Total records: 3731
Distinct categories: 14
In-stock vs out-of-stock:
 outofstock
False    3278
True      453
Name: count, dtype: int64


,name,discountpercent,discountedsellingprice
2619,Dukes Waffy Strawberry Wafers,51,22.0
2608,Dukes Waffy Chocolate Wafers,51,22.0
2615,Dukes Waffy Orange Wafers,51,22.0
1191,RRO Cheddar Block Cheese,50,147.0
1195,RRO Fresh Ricotta,50,137.0
1197,RRO Mozzarella Pizza Cheese,50,137.0
1643,Moi Soi Black Bean Sauce - Dip Spread Stir Fr...,50,140.0
1200,RRO Burrata Cheese,50,125.0
1326,RRO Mozzarella Pizza Cheese,50,137.0
1213,RRO Mascarpone Cheese,50,177.0


In [6]:
# 5) Plots (save to ../plots/)
if 'availableQuantity' in df.columns and 'discountedSellingPrice' in df.columns and 'category' in df.columns:
    df['est_revenue'] = df['availableQuantity'].fillna(0) * df['discountedSellingPrice'].fillna(0)
    rev = df.groupby('category', dropna=False)['est_revenue'].sum().sort_values(ascending=False).head(12)
    plt.figure(figsize=(10,6))
    sns.barplot(x=rev.values, y=rev.index, palette='viridis')
    plt.xlabel('Estimated Revenue (₹)')
    plt.title('Top Categories by Estimated Revenue')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'top_categories_revenue.png'))
    plt.close()
if 'price_per_gram' in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df['price_per_gram'].dropna(), bins=50)
    plt.xlabel('Price per gram (₹)')
    plt.title('Price per gram distribution')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'price_per_gram_hist.png'))
    plt.close()
if 'discountPercent' in df.columns and 'name' in df.columns:
    topd = df.sort_values('discountPercent', ascending=False).dropna(subset=['discountPercent']).head(12)
    plt.figure(figsize=(10,6))
    sns.barplot(x='discountPercent', y='name', data=topd, palette='magma')
    plt.xlabel('Discount %')
    plt.title('Top Discounted Products')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'top_discounts.png'))
    plt.close()
print('Plots saved to', PLOTS_DIR)

Plots saved to ../plots


/var/folders/35/kqk3kx196614x59dvppzrstw0000gn/T/ipykernel_49536/1333534347.py:17: UserWarning: Glyph 8377 (\N{INDIAN RUPEE SIGN}) missing from font(s) Arial.
  plt.tight_layout()
/var/folders/35/kqk3kx196614x59dvppzrstw0000gn/T/ipykernel_49536/1333534347.py:18: UserWarning: Glyph 8377 (\N{INDIAN RUPEE SIGN}) missing from font(s) Arial.
  plt.savefig(os.path.join(PLOTS_DIR, 'price_per_gram_hist.png'))


## Key takeaways
- High discounts exist in packaged snacks and dairy — verify promotional vs error pricing.
- Several high-MRP items are out of stock — prioritize replenishment for revenue recovery.
- Use `price_per_gram` for value-tier tagging in merchandising.